In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install faiss-cpu
!pip install imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.5 MB/s eta 0:00:00


In [6]:
import numpy as np
import pandas as pd
import faiss
from PIL import Image
import imagehash
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

In [7]:
EMB_DIR = "/content/drive/MyDrive/copydays-ndid/NDID_6000_image_data_set/embeddings"

emb = np.load(f"{EMB_DIR}/clip_embeddings.npy").astype("float32")
meta = pd.read_csv(f"{EMB_DIR}/clip_embeddings_meta.csv")

index = faiss.read_index("/content/drive/MyDrive/copydays-ndid/Nagarjuna/ndid_faiss.index")

In [8]:
K = 10
CLIP_THRESHOLD = 0.90
PHASH_THRESHOLD = 5

y_true = []
y_pred = []

In [ ]:
for i in tqdm(range(len(emb)), desc="F1 evaluation"):
    q = emb[i].reshape(1, -1)
    sims, idxs = index.search(q, K)

    sims = sims[0][1:]
    idxs = idxs[0][1:]

    q_group = meta.iloc[i]["group_id"]
    nn_groups = meta.iloc[idxs]["group_id"].values

    # ground truth
    y_true.append(int(q_group in nn_groups))

    # prediction
    pred = 0
    q_hash = imagehash.phash(Image.open(meta.iloc[i]["image_path"]))

    for j, sim in zip(idxs, sims):
        cand_hash = imagehash.phash(Image.open(meta.iloc[j]["image_path"]))
        if (q_hash - cand_hash) <= PHASH_THRESHOLD or sim >= CLIP_THRESHOLD:
            pred = 1
            break

    y_pred.append(pred)

F1 evaluation:   8%|▊         | 2491/30000 [15:23<2:40:52,  2.85it/s]

In [ ]:
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 Score :", f1_score(y_true, y_pred))